# PDD Mobility Scanner — Format Sensor Logger + smoothed CLIP predictions to dashboard XLSX

Combines two CSVs into the dashboard-compatible xlsx (`trail_data` + `waypoint_photos` sheets):

1. **Sensor Logger Location CSV** — `seconds_elapsed`, `location_latitude`, `location_longitude`, `location_altitude`, `location_speed`, `location_bearing`, …
2. **Smoothed CLIP/YOLO predictions CSV** — `frame`, `t_sec`, `clip_label_smoothed`, `clip_conf_smoothed`, `score_*_smoothed`, …

Each location row is matched to its nearest-or-prior smoothed frame by time, GPS values are forward-filled across the sparse sensor rows, and 5-second waypoints are assigned from `seconds_elapsed`. `surface_type` is taken from `clip_label_smoothed`. IMU/ToF/calibration columns are left blank since the Sensor Logger location export doesn't carry them.

## Step 1: Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import shutil
from google.colab import files

## Step 2: Upload both CSVs

Upload the Sensor Logger Location CSV and the smoothed predictions CSV. The notebook tells them apart by columns: the location CSV has `seconds_elapsed`, the smoothed CSV has `clip_label_smoothed`.

In [ ]:
uploaded = files.upload()
csvs = [f for f in uploaded.keys() if f.lower().endswith('.csv')]
assert len(csvs) >= 2, f'Expected 2 CSVs, got {len(csvs)}: {csvs}'

location_csv = smoothed_csv = None
for f in csvs:
    cols = pd.read_csv(f, nrows=0).columns
    if 'clip_label_smoothed' in cols:
        smoothed_csv = f
    elif 'seconds_elapsed' in cols:
        location_csv = f

assert location_csv,  'No CSV with `seconds_elapsed` column found (Sensor Logger Location).'
assert smoothed_csv, 'No CSV with `clip_label_smoothed` column found (CLIP/YOLO smoothed output).'
print(f'Location:  {location_csv}')
print(f'Smoothed:  {smoothed_csv}')

## Step 3: Load Sensor Logger location data

Forward-fill GPS columns so every row carries the most recent fix (raw sensor logger location rows are sparse — empty cells everywhere except the ~1 Hz GPS samples).

In [ ]:
loc = pd.read_csv(location_csv)
loc = loc.sort_values('seconds_elapsed').reset_index(drop=True)

GPS_COLS = [
    'location_latitude', 'location_longitude',
    'location_altitude', 'location_altitudeAboveMeanSeaLevel',
    'location_speed', 'location_bearing',
    'location_horizontalAccuracy', 'location_verticalAccuracy',
]
for c in GPS_COLS:
    if c in loc.columns:
        loc[c] = loc[c].replace(-1, np.nan).ffill()

print(f'Loaded {len(loc)} location rows, duration {loc["seconds_elapsed"].max():.1f}s')

## Step 4: Load smoothed predictions and snap each location row to its frame

Uses `merge_asof` so each `seconds_elapsed` row picks up `clip_label_smoothed` / `clip_conf_smoothed` from the nearest-or-prior YOLO frame.

In [ ]:
sm = pd.read_csv(smoothed_csv).sort_values('t_sec').reset_index(drop=True)
print(f'Loaded {len(sm)} smoothed prediction rows, t_sec span {sm["t_sec"].min():.2f}–{sm["t_sec"].max():.2f}s')

sm_keep = ['t_sec', 'clip_label_smoothed', 'clip_conf_smoothed']
sm_keep = [c for c in sm_keep if c in sm.columns]

merged = pd.merge_asof(
    loc.rename(columns={'seconds_elapsed': 't_sec'}),
    sm[sm_keep],
    on='t_sec',
    direction='nearest',
)
merged = merged.rename(columns={'t_sec': 'seconds_elapsed'})
print(f'Merged shape: {merged.shape}')

## Step 5: Assign waypoints (5-second windows)

In [ ]:
WAYPOINT_SEC = 5
merged['wp'] = np.maximum(0, np.floor(merged['seconds_elapsed'] / WAYPOINT_SEC).astype(int))
n_waypoints = merged['wp'].nunique()
print(f'{n_waypoints} waypoints over {len(merged)} rows '
      f'(avg {len(merged) / n_waypoints:.0f} rows per waypoint)')

## Step 6: Build the `trail_data` and `waypoint_photos` DataFrames

Same `TARGET_COLUMNS` schema as the original dashboard format. `surface_type` per row comes from `clip_label_smoothed`; per-waypoint surface for `waypoint_photos` is the most-common smoothed label across the waypoint's rows.

In [ ]:
TARGET_COLUMNS = [
    'ms', 'utc', 'wp',
    'ax', 'ay', 'az',
    'gvx', 'gvy', 'gvz',
    'gx', 'gy', 'gz',
    'qw', 'qx', 'qy', 'qz',
    'mx', 'my', 'mz',
    'tof_mm', 'tof_status',
    'lat', 'lng', 'alt', 'speed', 'hdg', 'sats', 'hdop',
    'cal_sys', 'cal_gyro', 'cal_accel', 'cal_mag',
    'image_file', 'surface_type', 'obstacle', 'substructure',
]

out_df = pd.DataFrame(index=merged.index, columns=TARGET_COLUMNS)

out_df['ms']  = (merged['seconds_elapsed'] * 1000).round().astype('Int64')
out_df['utc'] = merged['time'] if 'time' in merged.columns else ''
out_df['wp']  = merged['wp'].astype(int)

out_df['lat']   = merged.get('location_latitude')
out_df['lng']   = merged.get('location_longitude')
out_df['alt']   = merged.get('location_altitude')
out_df['speed'] = merged.get('location_speed')
out_df['hdg']   = merged.get('location_bearing')
out_df['hdop']  = merged.get('location_horizontalAccuracy')

out_df['image_file']   = out_df['wp'].map(lambda wp: f'img_{int(wp):04d}.jpg')
out_df['surface_type'] = merged.get('clip_label_smoothed', 'No surface detected').fillna('No surface detected')
out_df['obstacle']     = 'None detected'
out_df['substructure'] = 'None detected'

def _mode(series):
    s = series.dropna()
    return s.mode().iloc[0] if len(s) else 'No surface detected'

wp_surface = (
    merged.groupby('wp')['clip_label_smoothed']
    .apply(_mode)
    if 'clip_label_smoothed' in merged.columns
    else pd.Series(dtype=object)
)

waypoint_photos_df = (
    out_df[['wp', 'image_file']]
    .drop_duplicates(subset='wp')
    .sort_values('wp')
    .reset_index(drop=True)
)
waypoint_photos_df['surface_type']   = waypoint_photos_df['wp'].map(wp_surface).fillna('No surface detected')
waypoint_photos_df['obstacles']      = 'None detected'
waypoint_photos_df['infrastructure'] = 'None detected'

print(f'trail_data:      {len(out_df)} rows, {len(out_df.columns)} columns')
print(f'waypoint_photos: {len(waypoint_photos_df)} rows')
waypoint_photos_df.head(10)

## Step 7: Write XLSX and download

In [ ]:
out_dir = 'output'
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)
os.makedirs(out_dir)

xlsx_path = os.path.join(out_dir, 'trail_data.xlsx')
with pd.ExcelWriter(xlsx_path) as writer:
    out_df.to_excel(writer, sheet_name='trail_data', index=False)
    waypoint_photos_df.to_excel(writer, sheet_name='waypoint_photos', index=False)
print(f'Wrote {xlsx_path} (sheets: trail_data, waypoint_photos)')

files.download(xlsx_path)